# GPT reproduction first model

In [21]:
import torch

In [22]:
# 1. Read the input file.
import numpy as np

def generate_training_data_standard_distribution(size:int) -> str:
    data_int = np.random.randint(ord('a'),ord('z') + 1,size=size).tolist()
    return "".join([chr(x) for x in data_int])

def generate_training_data_special_pattern(size:int) -> str:
    data_int = []
    i = 0
    while i < size-1:
        # item = int(np.random.randint(ord('a'),ord('z') + 1))
        item = ord("a")
        data_int.append(item)
        i += 1
        if item == ord('a'):
            i += 1
            random2 = np.random.rand()
            if random2 <= 0.5:
                data_int.append(ord("b"))
            elif random2 > 0.5 and random2 <= 0.7:
                data_int.append(ord("c"))
            elif random2 > 0.7 and random2 <= 0.8:
                data_int.append(ord("d"))
            else:
                data_int.append(int(np.random.randint(ord('e'),ord('z') + 1)))
    if i != size:
        data_int.append(int(np.random.randint(ord('b'),ord('z') + 1)))
    return "".join([chr(x) for x in data_int])

generate_training_data = generate_training_data_special_pattern

# 2. Create the character-level vocabulary.

class Vocabulary:
    def __init__(self,data):
        self.vocabulary = sorted(list(set(list(training_data))))
        self.stoi = { c:i for i,c in enumerate(self.vocabulary) }
        self.itos = { i:c for i,c in enumerate(self.vocabulary) }

# 3. Create decode and encode function
    def encode_data(self,input:str) -> list:
        return [self.stoi[x] for x in input ]

    def decode_data(self,input:list) -> str:
        return "".join([self.itos[x] for x in input ])

    def size(self) -> int:
        return len(self.vocabulary)

training_data = generate_training_data(100000)
print(training_data[:100])

print("a",training_data.count("a"))
print("ab",training_data.count("ab"))
print("ac",training_data.count("ac"))
print("ad",training_data.count("ad"))
print("ad",training_data.count("ae"))
vocabulary = Vocabulary(training_data)
print(vocabulary.encode_data("bank"))
print(vocabulary.decode_data(vocabulary.encode_data("bank")))
# ----- PARAMETERS -----
EMBEDDING_DIM = vocabulary.size()


abafacafavabacavaqabacabacabaoadabatacabazabacababababatabababahababababacabababacaiabatafabababayac
a 50000
ab 24801
ac 10055
ad 5013
ad 423
[1, 0, 13, 10]
bank


In [23]:
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 2 # what is the maximum context length for predictions?

Ez a model most reprodukálja az lehetö legegyszerübb nyelvi modelt. Csak a loss kiszámítását tartalmazza, és a következö token generátor funkcióját

In [27]:
import torch
import torch.nn as nn
from torch.nn import functional as F


class LLM_Forward_Loss_Generator(nn.Module):  # What is the torch.nn.Module?

    def __init__(self, vocabulary: Vocabulary, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.token_embedding_table = nn.Embedding(vocabulary.size(), EMBEDDING_DIM)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx)  # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss


torch.manual_seed(1337)

network = LLM_Forward_Loss_Generator(vocabulary)
input = torch.tensor(
    [vocabulary.encode_data("a"), vocabulary.encode_data("a")], dtype=torch.long
)
target = torch.tensor(
    [vocabulary.encode_data("b"), vocabulary.encode_data("c")], dtype=torch.long
)
print(network.token_embedding_table(input).shape)

prediction, loss = network(input, target)
print(loss)
# print(prediction)

torch.Size([2, 1, 26])
tensor(3.9148, grad_fn=<NllLossBackward0>)


In [28]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(network.parameters(), lr=1e-3)

In [38]:
batch_size = 20
i = 0
while i * batch_size < len(training_data) / 2: # increase number of steps for good results...

    # sample a batch of data
    #for item in range(batch_size):
    xb = torch.stack([torch.tensor([vocabulary.encode_data([training_data[i+x] for x in range(batch_size)])])])
    yb = torch.stack([torch.tensor([vocabulary.encode_data([training_data[i+x] for x in range(batch_size)])])])
    print(xb,yb)
    break

    # evaluate the loss
    #logits, loss = m(xb, yb)
    #optimizer.zero_grad(set_to_none=True)
    #loss.backward()
    #optimizer.step()

print(loss.item())

tensor([[[ 0,  1,  0,  5,  0,  2,  0,  5,  0, 21,  0,  1,  0,  2,  0, 21,  0,
          16,  0,  1]]]) tensor([[[ 0,  1,  0,  5,  0,  2,  0,  5,  0, 21,  0,  1,  0,  2,  0, 21,  0,
          16,  0,  1]]])
3.9148471355438232
